# Aula 11: Análise de Dados com Pandas - MySQL `dbven`

Considere um cenário hipotético em que você foi contratado como analista de dados de uma **loja de produtos para ciclismo**. Os dados estão organizados nas tabelas **Clientes, Vendedores, Produtos, Vendas e ItensVenda** do banco `dbven`, o mesmo BD que já utilizamos na aula passada.

## Objetivo
Gerar **informações que possam apoiar decisões de vendas**, utilizando a biblioteca **pandas** para explorar, verificar e analisar os dados consultados no MySQL. 

## Pipeline do Processo de Análise de Dados
1. **Análise exploratória:** conhecer a estrutura dos dados, identificar suas características e obter estatísticas descritivas.
2. **Limpeza:** identificar e tratar possíveis inconsistências e valores ausentes, considerando sua influência nas análises.
3. **Análises:** realizar operações de seleção, agrupamento, ordenação e cálculos estatísticos para identificar padrões, tendências e informações relevantes.
4. **Recomendações:** interpretar os resultados obtidos e propor recomendações fundamentadas nos dados para apoiar a tomada de decisões.


## Desafio

Uma loja deseja entender **onde vende mais**, **quais produtos têm maior saída** e **como suas vendas variam por mês**.


## Passo 1: Trazer a base de dados para o Python e olhar o que tem nela

**1. Conexão:** a célula seguinte contém **exatamente** o código de conexão com o dbven.

**2. Consulta:** em seguida, a função `carregar_dados(connection)` utiliza a conexão recebida para consultar o esquema `dbven`. O cursor lê as tabelas e o Pandas as transforma em DataFrames.


In [1]:
# conexao.py
from getpass import getpass
from mysql.connector import connect, Error

def usar_conexao(acao):
    try:
        with connect(
            host="localhost",               # Endereço do servidor MySQL (localhost)
            user=input("Usuário: "),        # Solicita ao usuário o nome de login
            password=getpass("Senha: ")     # Solicita a senha de forma oculta
        ) as connection:
            print("Conexão estabelecida:", connection)
            return acao(connection)
    except Error as e:
        print("Ocorreu um erro ao conectar com o BD: ", e)

### Conceitos essenciais: DataFrame e Series

A consulta SQL recupera registros do MySQL. `pd.DataFrame()` organiza esses registros em **linhas e colunas**, para análise no Python. Cada linha de `tabela` representa **um item de venda**:

- **DataFrame:** estrutura bidimensional, semelhante a uma tabela.
- **Índice:** rótulo utilizado para identificar uma linha. Os nomes das colunas são rótulos da

In [2]:
import pandas as pd

# Consulta os dados de vendas no MySQL
def carregar_dados(connection):
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT v.IDVenda, v.Data AS `Data da Venda`,
                   c.Estado, p.Produto,
                   iv.Quantidade AS `Quantidade Vendida`,
                   iv.ValorTotal AS `Total de Vendas (R$)`
            FROM dbven.Vendas AS v
            JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
            JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
            JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto
        """)
        # Transforma os registros retornados pelo MySQL em um DataFrame do Pandas,
        # utilizando os nomes das colunas da consulta SQL
        return pd.DataFrame(
            cursor.fetchall(),
            columns=cursor.column_names
        )

# Consulta o banco de dados
tabela = usar_conexao(carregar_dados)

# Converte os valores monetários para números
tabela["Total de Vendas (R$)"] = pd.to_numeric(tabela["Total de Vendas (R$)"])

# Exibe os 10 primeiros registros
display(tabela.head(20))

Conexão estabelecida: <mysql.connector.connection_cext.CMySQLConnection object at 0x0000028C3F47A350>


,IDVenda,Data da Venda,Estado,Produto,Quantidade Vendida,Total de Vendas (R$)
0,5,2016-01-03,PB,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,8852.00
1,7,2016-01-06,SP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,7524.20
2,10,2016-01-06,PI,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,8852.00
3,11,2016-01-08,SE,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,7524.20
4,12,2016-01-09,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,8852.00
5,14,2016-01-12,TO,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,6019.36
6,22,2016-01-17,BA,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,7524.20
7,25,2016-01-19,SC,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,8852.00
8,26,2016-01-21,BA,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,8852.00
9,32,2016-01-24,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,2,14163.20


### Análise exploratória dos dados

Antes de calcular indicadores, precisamos saber **quantas linhas existem, quais colunas estão disponíveis e quais são seus tipos**. `head()` exibe os primeiros registros; `shape`, `columns`, `dtypes` e `index` descrevem a estrutura. A seleção `tabela["Produto"]` produz uma Series; já `tabela[["Estado", "Produto"]]` produz um DataFrame.

In [ ]:
# Quantidade de linhas e colunas
print("Dimensões (linhas, colunas):", tabela.shape)

# Uma coluna = Series; duas colunas = DataFrame
display(tabela["Produto"].head())
display(tabela[["Estado", "Produto"]].head())

Dimensões (linhas, colunas): (468, 6)


0    Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
1    Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
2    Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
3    Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
4    Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
Name: Produto, dtype: object

,Estado,Produto
0,PB,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
1,SP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
2,PI,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
3,SE,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
4,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...


### Resumo estatístico dos dados numéricos (uma visão geral da base)

`describe()` apresenta contagem, média, desvio padrão, mínimo, quartis e máximo. O resultado ajuda a conhecer escalas e valores extremos.


In [ ]:
# Calcula as estatísticas descritivas das colunas numéricas.
# count = quantidade de valores não nulos; mean = média; std = desvio padrão.
# min e max = valores mínimo e máximo.
# 25%, 50% e 75% = percentis dos dados.
tabela[["Quantidade Vendida", "Total de Vendas (R$)"]].describe()

# SQL equivalente:
# SELECT
#     COUNT(iv.Quantidade) AS count_quantidade,
#     AVG(iv.Quantidade) AS mean_quantidade,
#     STDDEV_SAMP(iv.Quantidade) AS std_quantidade,
#     MIN(iv.Quantidade) AS min_quantidade,
#     MAX(iv.Quantidade) AS max_quantidade,
#     COUNT(iv.ValorTotal) AS count_valor,
#     AVG(iv.ValorTotal) AS mean_valor,
#     STDDEV_SAMP(iv.ValorTotal) AS std_valor,
#     MIN(iv.ValorTotal) AS min_valor,
#     MAX(iv.ValorTotal) AS max_valor
# FROM dbven.Vendas AS v
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
# JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto;

### Seleção de linhas e colunas: `loc[]` e `iloc[]`

- `loc[linhas, colunas]` seleciona pelos **rótulos** dos índices e pelos nomes das colunas.
- `iloc[linhas, colunas]` seleciona pelas **posições numéricas** (começando em zero).

Observe que o índice padrão é numérico; se ele for alterado, `loc[0]` e `iloc[0]` não terão necessariamente o mesmo resultado. No exemplo, `loc` utiliza os rótulos existentes na tabela.

In [8]:
# Duas colunas de cinco registros, usando seus rótulos
display(tabela.loc[tabela.index[:5], ["Estado", "Produto"]])

# Três primeiras linhas pelas posições 0, 1 e 2
display(tabela.iloc[0:3])


,Estado,Produto
0,PB,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
1,SP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
2,PI,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
3,SE,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...
4,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...


,IDVenda,Data da Venda,Estado,Produto,Quantidade Vendida,Total de Vendas (R$)
0,5,2016-01-03,PB,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,8852.0
1,7,2016-01-06,SP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,7524.2
2,10,2016-01-06,PI,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,8852.0


### Limpeza da base de dados

- `isnull().sum()` → conta os valores ausentes por coluna.
- `dropna()` → elimina linhas com ao menos um valor ausente.
- `dropna(axis=1)` → elimina colunas com algum valor ausente.
- `dropna(how="all")` → elimina apenas linhas totalmente vazias.
- `dropna(subset=[...])` → considera uma coluna específica.

Atenção: **não elimine dados automaticamente**; avalie se o campo é necessário para a pergunta que está sendo respondida.


In [9]:
# Conta os valores ausentes em cada coluna.
print(tabela.isnull().sum())

# Exemplos: novas tabelas, sem modificar o DataFrame original.
tabela_sem_nulos = tabela.dropna()                 # Remove linhas com algum valor nulo.
tabela_sem_colunas_nulas = tabela.dropna(axis=1)  # Remove colunas com algum valor nulo.
tabela_drop_all_null = tabela.dropna(how="all")  # Remove linhas inteiramente nulas.
tabela_sem_qtd_nula = tabela.dropna(subset=["Quantidade Vendida"])

# Atenção: eliminar linhas pode alterar as análises. Verifique antes.
print("Linhas antes:", len(tabela))
print("Linhas após dropna():", len(tabela_sem_nulos))


IDVenda                 0
Data da Venda           0
Estado                  0
Produto                 0
Quantidade Vendida      0
Total de Vendas (R$)    0
dtype: int64
Linhas antes: 468
Linhas após dropna(): 468


### Filtragem: selecionar registros que atendem a uma condição

Assim como `WHERE` no SQL, uma **condição booleana** no Pandas permite manter somente as linhas desejadas. A expressão `tabela["Estado"] == "RS"` produz valores `True` ou `False` para cada linha; ao utilizá-la dentro de `tabela[...]`, recuperamos somente as linhas com `True`.


In [10]:
# SQL equivalente (sobre as mesmas tabelas da consulta inicial):
# SELECT v.IDVenda, c.Estado, p.Produto, iv.Quantidade, iv.ValorTotal
# FROM dbven.Vendas AS v
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
# JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto
# WHERE c.Estado = 'RS';

# Cria um novo DataFrame somente com os registros do RS
vendas_rs = tabela[tabela["Estado"] == "RS"]
display(vendas_rs.head(10))
print("Registros do RS:", len(vendas_rs))

,IDVenda,Data da Venda,Estado,Produto,Quantidade Vendida,Total de Vendas (R$)
34,127,2016-04-21,RS,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,5417.42
35,161,2016-05-22,RS,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,1,7524.20
62,93,2016-03-17,RS,Bicicleta Altools Stroll Aro 26 Freio À Disco ...,1,7038.77
189,161,2016-05-22,RS,Bicicleta Gometws Endorphine 7.3 - Shimano Alu...,1,2268.99
331,161,2016-05-22,RS,Luva De Ciclismo - Gometws Sports,1,188.00
384,127,2016-04-21,RS,Bermuda Predactor 3Xu Pro,1,115.00
391,161,2016-05-22,RS,Bermuda Predactor 3Xu Pro,1,82.80
455,161,2016-05-22,RS,Camiseta Predactor 3Xu Multiplied,1,114.75


Registros do RS: 8


## Passo 2: Visão geral sobre a base de dados

Qual é o faturamento total, quantas vendas foram feitas e qual e qual o valor médio por venda? Distinguir **item de venda** de **venda individual**.


In [11]:
# Calcula o faturamento total
vendas_total = tabela["Total de Vendas (R$)"].sum()
print(f"Faturamento total: R$ {vendas_total:,.2f}")

# SQL equivalente:
# SELECT SUM(iv.ValorTotal) AS FaturamentoTotal
# FROM dbven.ItensVenda AS iv
# JOIN dbven.Vendas AS v ON iv.IDVenda = v.IDVenda
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto;


# Agrupa os itens para calcular o valor de cada venda
vendas = tabela.groupby("IDVenda")["Total de Vendas (R$)"].sum()

# SQL equivalente:
# SELECT v.IDVenda, SUM(iv.ValorTotal) AS TotalVenda
# FROM dbven.Vendas AS v
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
# JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto
# GROUP BY v.IDVenda;


# Calcula a quantidade de vendas e o valor médio por venda
print("Quantidade de vendas:", len(vendas))
print(f"Valor médio por venda: R$ {vendas.mean():,.2f}")

# SQL equivalente:
# SELECT COUNT(*) AS QuantidadeVendas,
#        AVG(TotalVenda) AS ValorMedioVenda
# FROM (
#     SELECT v.IDVenda, SUM(iv.ValorTotal) AS TotalVenda
#     FROM dbven.Vendas AS v
#     JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
#     JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
#     JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto
#     GROUP BY v.IDVenda
# ) AS vendas_agrupadas;


Faturamento total: R$ 1,532,637.70
Quantidade de vendas: 198
Valor médio por venda: R$ 7,740.59


## Passo 3: Começar a análise TOP → DOWN

Vamos começar pelo **estado cadastrado do cliente**, comparando a receita e as quantidades vendidas por UF.


### Agrupamentos: do registro individual ao resumo

A mesma UF aparece em muitas linhas da tabela, pois há vários itens de vendas. Para responder **quanto foi faturado em cada estado**, o Pandas precisa reunir as linhas com o mesmo estado e somar seus valores.

1. `groupby("Estado")` identifica os grupos.
2. `sum()` soma o valor de cada grupo.
3. `sort_values(..., ascending=False)` ordena do maior para o menor.

No SQL, essas etapas correspondem principalmente a `GROUP BY`, `SUM()` e `ORDER BY ... DESC`.

In [12]:
# 1) Seleciona as colunas necessárias
dados_estado = tabela[["Estado", "Total de Vendas (R$)"]]
display(dados_estado.head(5))

# 2) Agrupa e soma os valores
resumo_estado = dados_estado.groupby("Estado").sum()
display(resumo_estado)

# 3) Ordena o resultado
display(resumo_estado.sort_values("Total de Vendas (R$)", ascending=False))

,Estado,Total de Vendas (R$)
0,PB,8852.0
1,SP,7524.2
2,PI,8852.0
3,SE,7524.2
4,AP,8852.0


,Total de Vendas (R$)
Estado,
AC,34444.47
AM,111926.32
AP,65529.67
BA,209101.23
ES,164332.98
MA,62741.99
MG,27856.50
MT,76263.55
PA,61378.99


,Total de Vendas (R$)
Estado,
BA,209101.23
ES,164332.98
SC,150540.06
AM,111926.32
PB,105966.13
RJ,97397.26
SP,85101.00
TO,76474.03
MT,76263.55


In [13]:
# Agrupa as vendas por estado e calcula o faturamento total.
vendas_por_estado = tabela[["Estado", "Total de Vendas (R$)"]].groupby("Estado").sum()

# Ordena os estados pelo faturamento, do maior para o menor.
vendas_por_estado = vendas_por_estado.sort_values("Total de Vendas (R$)", ascending=False)

display(vendas_por_estado)

# SQL equivalente:
# SELECT c.Estado, SUM(iv.ValorTotal) AS Faturamento
# FROM dbven.Vendas AS v
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
# GROUP BY c.Estado
# ORDER BY Faturamento DESC;


# Agrupa as vendas por estado e soma as unidades vendidas.
quantidade_por_estado = tabela.groupby("Estado")["Quantidade Vendida"].sum()

# Ordena os estados pela quantidade vendida.
display(quantidade_por_estado.sort_values(ascending=False))

# SQL equivalente:
# SELECT c.Estado, SUM(iv.Quantidade) AS QuantidadeVendida
# FROM dbven.Vendas AS v
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
# GROUP BY c.Estado
# ORDER BY QuantidadeVendida DESC;


,Total de Vendas (R$)
Estado,
BA,209101.23
ES,164332.98
SC,150540.06
AM,111926.32
PB,105966.13
RJ,97397.26
SP,85101.00
TO,76474.03
MT,76263.55


Estado
ES    66
SC    56
BA    48
PB    39
TO    34
AM    31
MT    29
RJ    28
MA    27
PA    24
SP    24
AP    20
PE    18
RN    18
AC    16
PR    15
MG    12
PI    11
RS     8
SE     3
Name: Quantidade Vendida, dtype: int64

## Passo 4: Buscar por mais detalhex para entender

Após identificar os estados, detalhamos os **produtos** em cada um deles. Assim avançamos dos totais gerais para perguntas mais específicas.


In [14]:
# Passo 4: explorar quais produtos foram comprados em cada estado.
vendas_por_produto = tabela[["Estado", "Produto", "Total de Vendas (R$)"]]
display(vendas_por_produto.head(10))

# SQL equivalente:
# SELECT c.Estado, p.Produto, iv.ValorTotal AS Faturamento
# FROM dbven.Vendas AS v
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
# JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto
# LIMIT 10;


,Estado,Produto,Total de Vendas (R$)
0,PB,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,8852.00
1,SP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,7524.20
2,PI,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,8852.00
3,SE,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,7524.20
4,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,8852.00
5,TO,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,6019.36
6,BA,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,7524.20
7,SC,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,8852.00
8,BA,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,8852.00
9,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,14163.20


### Índice hierárquico (MultiIndex)

Ao agrupar por **duas colunas** (`Estado` e `Produto`), essas duas informações passam a compor o índice do resultado. Isso é um **MultiIndex**. Basta compreender que cada linha do resultado representa uma combinação de estado e produto. 

In [15]:
# Agrupamento por estado e produto, somando a receita dos itens.
vendas_por_produto = (
    tabela[["Estado", "Produto", "Total de Vendas (R$)"]]
    .groupby(["Estado", "Produto"])
    .sum()
)
display(vendas_por_produto)

# Para exibir todas as linhas temporariamente, descomente:
# with pd.option_context("display.max_rows", None, "display.max_columns", None):
#     display(vendas_por_produto)


# SQL equivalente:
# SELECT c.Estado, p.Produto, SUM(iv.ValorTotal) AS Faturamento
# FROM dbven.Vendas AS v
# JOIN dbven.Clientes AS c ON v.IDCliente = c.IDCliente
# JOIN dbven.ItensVenda AS iv ON v.IDVenda = iv.IDVenda
# JOIN dbven.Produtos AS p ON iv.IDProduto = p.IDProduto
# GROUP BY c.Estado, p.Produto;


Total de Vendas (R$)
Estado Produto                                                                 
AC     Bermuda Predactor 3Xu Pro                                         207.00
       Bicicleta Altools Stroll Aro 26 Freio À Disco 2...               8280.90
       Bicicleta Aro 29 Mountain Bike Endorphine 6.3 -...               8852.00
       Bicicleta Gometws Endorphine 7.3 - Shimano Alum...               5101.52
       Bicicleta Gts Advanced 1.0 Aro 29 Freio Disco C...               4255.00
...                                                                         ...
TO     Bicicleta Gts Advanced 1.0 Aro 29 Freio Disco C...               3616.75
       Bicicleta Trinc Câmbios Shimano Aro 29 Freio A ...               4686.70
       Camiseta Predactor 3Xu Multiplied                                 540.00
       Capacete Gometws Endorphine 2.0                                   616.44
       Luva De Ciclismo - Gometws Sports                                 644.84

[171 rows x 1 columns]

### Encontrar o maior resultado de cada grupo: `idxmax()` e `loc[]`

O primeiro agrupamento soma as unidades do **mesmo produto em cada estado**. Depois, `idxmax()` encontra o **rótulo da linha** com a maior quantidade de cada estado. Finalmente, `loc[]` recupera as linhas e colunas correspondentes.

**Atenção:** em caso de empate, `idxmax()` retorna a primeira ocorrência do valor máximo, não todos os empatados.

In [19]:
# Produto mais vendido em unidades por UF.
# Primeiro somamos unidades por Estado e Produto: um produto pode aparecer em vários itens.
# O parâmetro as_index=False é utilizado no método groupby() do Pandas para indicar que as 
# colunas utilizadas no agrupamento devem continuar como colunas normais do DataFrame, em 
# vez de se tornarem índices.
quantidades_produto = (
    tabela.groupby(["Estado", "Produto"], as_index=False)["Quantidade Vendida"].sum()
)

# idxmax() localiza a linha do maior total de unidades em cada Estado.
maiores_quantidades = quantidades_produto.loc[
    quantidades_produto.groupby("Estado")["Quantidade Vendida"].idxmax(),
    ["Estado", "Produto", "Quantidade Vendida"]
]
display(maiores_quantidades)


,Estado,Produto,Quantidade Vendida
6,AC,Camiseta Predactor 3Xu Multiplied,4
10,AM,Bicicleta Altools Stroll Aro 26 Freio À Disco ...,6
20,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,4
28,BA,Bicicleta Altools Stroll Aro 26 Freio À Disco ...,11
41,ES,Bicicleta Gometws Endorphine 7.3 - Shimano Alu...,11
47,MA,Bermuda Predactor 3Xu Pro,5
57,MG,Bermuda Predactor 3Xu Pro,2
64,MT,Bermuda Predactor 3Xu Pro,4
79,PA,Luva De Ciclismo - Gometws Sports,6
88,PB,Capacete Gometws Endorphine 2.0,6


### Quantidade vendida não é o mesmo que faturamento

O produto com mais **unidades vendidas** pode ser diferente do produto que gerou o maior **valor em reais**. Compare os dois resultados. Nas próximas linhas, o objeto `vendas` foi construído com `groupby("IDVenda")`.

In [21]:
# Produto que mais gerou receita em cada estado (somando todos os itens).
receita_produto = (
    tabela.groupby(["Estado", "Produto"], as_index=False)["Total de Vendas (R$)"].sum()
)
maiores_receitas = receita_produto.loc[
    receita_produto.groupby("Estado")["Total de Vendas (R$)"].idxmax(),
    ["Estado", "Produto", "Total de Vendas (R$)"]
]
display(maiores_receitas)


,Estado,Produto,Total de Vendas (R$)
2,AC,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,8852.00
10,AM,Bicicleta Altools Stroll Aro 26 Freio À Disco ...,42711.05
20,AP,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,31867.20
28,BA,Bicicleta Altools Stroll Aro 26 Freio À Disco ...,90123.80
38,ES,Bicicleta Altools Stroll Aro 26 Freio À Disco ...,40116.36
49,MA,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,24475.78
61,MG,Bicicleta Trinc Câmbios Shimano Aro 29 Freio A...,14167.30
66,MT,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,24378.40
76,PA,Bicicleta Aro 29 Mountain Bike Endorphine 6.3 ...,30096.80
81,PB,Bicicleta Altools Stroll Aro 26 Freio À Disco ...,25762.80


### Análise temporal: faturamento por mês

`pd.to_datetime()` prepara as datas para operações temporais. `.dt.to_period("M")` representa cada data pelo seu **ano e mês**; `groupby()` e `sum()` permitem calcular o faturamento mensal.

Como `vendas` contém apenas o ID e o total de cada venda, a evolução mensal é calculada a partir de `tabela`, que já possui a data de cada item. Assim, não é necessário consultar novamente o banco.

In [27]:
# 1) Cria a coluna AnoMes com o ano e o mês de cada venda.
tabela["AnoMes"] = tabela["Data da Venda"].dt.to_period("M")

# 2) Agrupa as vendas por mês e soma o faturamento.
mensal = (
    tabela.groupby("AnoMes", as_index=False)["Total de Vendas (R$)"]
    .sum()
    .sort_values("AnoMes")
)

# 3) Exibe o faturamento mensal.
display(mensal)

,AnoMes,Total de Vendas (R$)
0,2016-01,350913.03
1,2016-02,231641.91
2,2016-03,239162.98
3,2016-04,248414.58
4,2016-05,217074.94
5,2016-06,235983.82
6,2016-07,9446.44
